In [25]:
import os
from dotenv import load_dotenv

def load_environment_variables():
    """
    Loads the environment variables from .env.
    Loads OpenAI api key and target model name used in the experiment
    New environment variables as necessary will be appended
    """

    PROJECT_ROOT = Path.cwd().parent
    ENV_FILE = PROJECT_ROOT / ".env"

    if ENV_FILE.exists():
        print(
            f".env file located at: {ENV_FILE.resolve()}")
        
        load_dotenv()
        env = {
                "api_key": os.getenv("OPENAI_API_KEY"),
                "model_name": os.getenv("OPENAI_MODEL_NAME")
            }

    else:
        print(
            f".env not found at: {ENV_FILE.resolve()}"
        )
        env = {
            "api_key": None,
            "model_name": None
        }

    return env

In [26]:
from openai import OpenAI

env = load_environment_variables()

client = OpenAI(
    api_key=env['api_key']
)

response = client.chat.completions.create(
    model=env['model_name'],
    messages=[
        {
            "role": "user",
            "content": "what is RAG?"
        }
    ]
)

print(response.choices[0].message.content)

.env file located at: D:\Bits-lab\eduassist-llmops-prompt-evaluation\.env


2026-11-01 06:56:17,615 | INFO | HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 200 OK"


RAG stands for **Retrieval-Augmented Generation**. It is a technique used primarily in natural language processing (NLP) that combines pre-trained language models with external information retrieval systems to generate more accurate and contextually relevant responses.

### How RAG Works:
1. **Retrieval:** When given a query or prompt, the system first retrieves relevant documents or pieces of information from a large external corpus (like Wikipedia, databases, or any other text source).
2. **Augmentation:** The retrieved information is then used to augment the input.
3. **Generation:** A language generation model (such as a transformer-based model like GPT) uses both the original input and the retrieved documents to generate a response.

### Benefits of RAG:
- **Improved accuracy:** By grounding answers in factual retrieved content, it reduces the hallucination problem where language models produce plausible-sounding but incorrect information.
- **Up-to-date knowledge:** Allows models

In [27]:
response = client.chat.completions.create(
    model=env['model_name'],
    messages=[
        {
            "role": "user",
            "content": "what is RAG in Natural Language Processing?"
        }
    ]
)

print(response.choices[0].message.content)

2026-11-01 06:56:45,994 | INFO | HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 200 OK"


In Natural Language Processing (NLP), **RAG** stands for **Retrieval-Augmented Generation**. It is a type of model architecture that combines the strengths of retrieval-based methods and generative models to improve tasks like question answering, summarization, and open-domain dialogue.

### How RAG works:
1. **Retrieval Component:** Given an input (e.g., a question), the model first retrieves relevant documents or passages from a large external knowledge base or corpus. This step uses techniques similar to information retrieval, often involving dense vector embeddings for similarity search.

2. **Generative Component:** The retrieved documents are then fed as additional context to a generative language model (such as a Transformer-based model like BART or T5), which generates the final output based on both the input and the retrieved information.

### Why use RAG?
- **Access to large external knowledge:** Instead of storing all knowledge in the model parameters (which can be limited b

In [28]:
questions = [
    "what is overfitting in machine learning?",
    "Explain gradient descent in simple terms.",
    "what is neural network?",
    "Difference between supervised and unsupervised learning?",
    "Explain backpropagation in neural networks."
]

In [29]:
PROMT_V1 = """
Answer the following student questions clearly.
Question: (Question)
"""

In [30]:
import time
import pandas as pd

def evaluate_prompt(
        questions,
        prompt_template,
        prompt_version,
        model
):
    results = []

    for idx, question in enumerate(questions, start=1):

        prompt = prompt_template.format(
            question=question
        )

        start_time = time.time()

        response = client.chat.completions.create(
            model=model,
            messages=[
                {
                    "role": "user",
                    "content": prompt
                }
            ]
        )

        end_time = time.time()

        latency = end_time - start_time

        results.append(
            {
                "question_id": idx,
                "prompt_version": prompt_version,
                "model": "gpt-4.1-mini",
                "latency_seconds": latency,
                "input_tokens": response.usage.prompt_tokens,
                "output_tokens": response.usage.completion_tokens,
                "total_tokens": response.usage.total_tokens,
                "response_status": "success"
            }
        )

    return pd.DataFrame(results)

In [32]:
df_v1 = evaluate_prompt(
    questions,
    PROMT_V1,
    "v1",
    env['model_name']
)

print("Statistics of V1 evaluation")
print(df_v1['latency_seconds'].describe())

latency_stats_v1 = df_v1['latency_seconds'].agg(
    ['mean', 'median', "min", 'max', 'std']
).to_frame("value")

print("Selected parameters")
print(latency_stats_v1)

2026-11-01 06:58:33,131 | INFO | HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 200 OK"
2026-11-01 06:58:33,831 | INFO | HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 200 OK"
2026-11-01 06:58:34,450 | INFO | HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 200 OK"
2026-11-01 06:58:35,620 | INFO | HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 200 OK"
2026-11-01 06:58:36,462 | INFO | HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 200 OK"


Statistics of V1 evaluation
count    5.000000
mean     0.866144
std      0.224785
min      0.617620
25%      0.696668
50%      0.848708
75%      0.992975
max      1.174748
Name: latency_seconds, dtype: float64
Selected parameters
           value
mean    0.866144
median  0.848708
min     0.617620
max     1.174748
std     0.224785
